# Network Anomaly Detection Trained on Home Network Traffic
Trains an IsolationForest on your home network traffic.
No attack labels needed — the model learns what *normal* looks like and flags everything else.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from google.colab import files

In [ ]:
# Upload your home traffic CSV captured with src/capture_home.py
print('Upload home_traffic.csv')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
DATA_PATH = f'/content/{fname}'
print(f'Loaded: {DATA_PATH}')

In [ ]:
DROP_COLS = [
    'Flow ID', 'Src IP', 'Dst IP', 'Timestamp',
    'Traffic Type', 'Traffic Subtype', 'Label', 'label',
    'src_ip', 'dst_ip', 'timestamp', 'flow_id'
]

df = pd.read_csv(DATA_PATH, low_memory=False, on_bad_lines='skip')
print(f'Raw shape: {df.shape}')

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
df = df.replace([np.inf, -np.inf], np.nan).dropna()
df = df.select_dtypes(include=[np.number])

print(f'After cleaning: {df.shape}')
print(f'Features: {list(df.columns)}')

In [ ]:
# EDA — feature distributions
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
top_cols = df.std().sort_values(ascending=False).head(20).index
for ax, col in zip(axes.flatten(), top_cols):
    df[col].clip(df[col].quantile(0.01), df[col].quantile(0.99)).hist(ax=ax, bins=40, color='steelblue')
    ax.set_title(col, fontsize=8)
    ax.set_xlabel('')
plt.suptitle('Top 20 Features by Variance (Home Traffic)', fontsize=14)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100)
plt.show()

In [ ]:
# Correlation heatmap (top 20 features)
top20 = df.std().sort_values(ascending=False).head(20).index
plt.figure(figsize=(14, 12))
sns.heatmap(df[top20].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation — Home Traffic')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100)
plt.show()

In [ ]:
# PCA — visualize traffic clusters
scaler_eda = StandardScaler()
X_eda = scaler_eda.fit_transform(df)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_eda)

plt.figure(figsize=(10, 7))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=5, color='steelblue')
plt.title('PCA — Home Traffic (2D)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.tight_layout()
plt.savefig('pca_plot.png', dpi=100)
plt.show()
print(f'Total variance explained: {pca.explained_variance_ratio_.sum():.1%}')

In [ ]:
feature_cols = list(df.columns)
scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols].values)
print(f'Training matrix: {X.shape}')

In [ ]:
# contamination: expected fraction of anomalies in YOUR traffic.
# 0.01 = 1% of your capture is expected to be unusual (safe default)
model = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
)
print('Training IsolationForest...')
model.fit(X)
print('Done.')

In [ ]:
# Check anomaly scores on training data
scores = model.decision_function(X)  # higher = more normal
preds = model.predict(X)             # 1 = normal, -1 = anomaly

n_anomalies = (preds == -1).sum()
print(f'Flows in training data: {len(preds)}')
print(f'Flagged as anomaly: {n_anomalies} ({n_anomalies/len(preds):.1%})')

plt.figure(figsize=(10, 4))
plt.hist(scores, bins=60, color='steelblue', edgecolor='none')
plt.axvline(0, color='red', linestyle='--', label='Decision boundary')
plt.title('Anomaly Score Distribution (Home Traffic)')
plt.xlabel('Score (higher = more normal)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=100)
plt.show()

In [ ]:
artifacts = {
    'model': model,
    'scaler': scaler,
    'feature_cols': feature_cols,
}
joblib.dump(artifacts, 'model.pkl')
print('Saved model.pkl')

files.download('model.pkl')
files.download('feature_distributions.png')
files.download('score_distribution.png')
print('Place model.pkl in the models/ folder.')